# Week 2 — Day 3: Tool Calling Basics
### CalderR Agentic AI Engineering Internship · Wednesday 1 July 2026

**Today's stack:** Python · LangChain (`langchain-core`, `langchain-groq`) · Groq API (`openai/gpt-oss-120b`) · Pydantic v2

**Scope (from the Week 2 brief):**
- Theory: tool/function-calling lifecycle, tool schemas (OpenAI format), `ToolMessage`, tool selection logic, parallel tool calls, tool chaining
- LangChain mechanics: `@tool` decorator, `BaseTool` class, `StructuredTool`, tool binding to chat models (`bind_tools()`)
- Reading: OpenAI function-calling docs, Groq tool-calling reference, LangChain Tools documentation
- Practice: build a first tool-calling agent (weather lookup + calculator)
- Lab 2.2 (Wednesday's applied piece): build a 5-tool agent -- `search_db`, `calculate`, `format_date`, `convert_currency`, `summarize_text` -- test each tool independently, then together

**Continuity from Day 2:** yesterday, a Pydantic model described the *shape of an extracted answer*. Today, a Pydantic-backed schema describes the *shape of a function call* -- same validation mechanics, different purpose. A tool's `args_schema` is validated exactly the way `JobPosting` was validated; the only new piece is that a successful validation now triggers an actual Python function call instead of just handing you a clean object.

**Model compatibility note, carried forward and re-verified for today specifically:** plain tool binding via `ChatGroq(...).bind_tools([...])` on `openai/gpt-oss-120b` is the well-supported, Groq-day-zero-launch path -- distinct from the `create_agent`/`with_structured_output` edge cases flagged on Day 2, which involve different internal code paths (`ProviderStrategy`/`ToolStrategy` response-format machinery, not plain `bind_tools()`). Today's lab uses `bind_tools()` directly, which is the mechanism Groq's own docs demonstrate for this model.

---

## 0. Environment Setup

In [1]:
# requirements-day3.txt
# langchain-core==0.3.51
# langchain-groq==0.2.3
# groq==0.32.0
# pydantic==2.9.2
# python-dotenv==1.0.1

!pip install -q --upgrade langchain-core langchain-groq groq pydantic python-dotenv



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
from datetime import datetime, date
from decimal import Decimal
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError
from langchain_core.tools import tool, StructuredTool, BaseTool
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage
from langchain_groq import ChatGroq

load_dotenv()

API_KEY = os.environ.get("GROQ_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "GROQ_API_KEY not found. Create a `.env` file next to this notebook "
        "with GROQ_API_KEY=gsk_... and keep `.env` out of git."
    )

MODEL_NAME = "openai/gpt-oss-120b"
llm = ChatGroq(model=MODEL_NAME, api_key=API_KEY, temperature=0)

print(f"ChatGroq client ready. Model: {MODEL_NAME}")


ChatGroq client ready. Model: openai/gpt-oss-120b


## 1. The Tool-Calling Lifecycle

### 1.1 What "tool calling" actually is (and isn't)

A model does **not** execute your Python function. It never touches your runtime, your filesystem, or your network. What actually happens:

1. You describe a function to the model as a **schema**: name, natural-language description, parameter names/types/descriptions. This schema -- not the function body -- is what gets sent to the API.
2. The model, given a user message and the available tool schemas, decides whether answering requires a tool, and if so, which one and with what arguments. It outputs this decision as **structured data** (a `tool_call`: name + arguments), not as code, and not as prose describing what it would do.
3. **Your code** (or a LangChain agent loop) reads that structured decision, looks up the *actual* Python function by name, and calls it with the model-provided arguments.
4. Your code packages the function's return value into a `ToolMessage` and appends it to the conversation.
5. The model is called again, now with the tool's result in context, and produces a final natural-language answer (or, in a multi-step task, decides to call another tool).

The model is the **decision-maker**; your code is the **executor**. This separation is the single most important conceptual fact about tool calling -- it is also the entire reason tool calling is *safer* than letting a model directly execute arbitrary code: every tool call passes through code you wrote and can validate, rate-limit, log, or refuse, before anything real happens.

### 1.2 The lifecycle as a diagram, in words

```
User message
   │
   ▼
Model (with tool schemas bound) ── decides ──► "I need calculate(a=12, b=7, op='multiply')"
   │                                                        │
   │  (model emits a tool_call, NOT a function execution)   │
   ▼                                                        ▼
Your code receives the tool_call          Your code looks up the REAL
                                           Python function `calculate`
                                                        │
                                                        ▼
                                           Your code EXECUTES calculate(12, 7, 'multiply')
                                                        │
                                                        ▼
                                           Result (e.g. 84) wrapped in a ToolMessage
                                                        │
                                                        ▼
ToolMessage appended to conversation history
                                                        │
                                                        ▼
Model called AGAIN, now sees the tool's result, produces final answer:
"12 multiplied by 7 is 84."
```

### 1.3 Tool schemas -- the OpenAI function-calling format

Groq's API is OpenAI-compatible, so tool schemas follow the same JSON Schema-based structure OpenAI standardized. A tool, at the wire level, looks like this regardless of which LangChain abstraction you used to generate it:

In [ ]:
example_tool_schema = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Performs a single arithmetic operation between two numbers.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "number", "description": "The first operand"},
                "b": {"type": "number", "description": "The second operand"},
                "operation": {
                    "type": "string",
                    "enum": ["add", "subtract", "multiply", "divide"],
                    "description": "Which arithmetic operation to perform",
                },
            },
            "required": ["a", "b", "operation"],
        },
    },
}

print(json.dumps(example_tool_schema, indent=2))


**The two things that matter most in this schema, in order of how often beginners get them wrong:**

1. **`description` is the model's *only* window into what your function does.** It never sees your function's source code. A vague description ("does math") leads to wrong tool selection; a precise one ("performs exactly one arithmetic operation between two numbers") leads to correct selection. This is the exact same principle as Day 1's prompt-engineering material -- you are still writing a prompt, it's just embedded in a schema field instead of a chat message.
2. **`enum` constrains valid values at the schema level**, not just in your function's internal validation. Constraining `operation` to four literal strings means the model is far less likely to invent a fifth ("modulo") that your function doesn't handle -- though it can still happen, which is exactly why your function needs its own validation too (defense in depth, not "the schema replaces validation").

### 1.4 `ToolMessage` -- how results get back to the model

A `ToolMessage` is a distinct message type (alongside `HumanMessage`, `AIMessage`, `SystemMessage`) that carries a tool's output back into the conversation, tagged with the `tool_call_id` that requested it -- this ID linkage is what lets the model correctly associate "the result I'm seeing" with "the specific call I made," which matters once more than one tool call happens in the same turn (Section 1.6).

### 1.5 Tool selection logic -- how the model decides

This is *not* a separate mechanism from normal next-token prediction -- the model has simply been fine-tuned (during its own training, by its provider) to recognize when a user's stated need matches a bound tool's description closely enough to emit a tool_call instead of a prose response. Three practical levers you control:

- **`tool_choice="auto"`** (default) -- model decides freely whether to call a tool, and which one.
- **`tool_choice="required"`** -- model must call *some* tool; useful when you know the task always needs one and want to eliminate the "model just answered in prose instead" failure mode.
- **`tool_choice={"type": "function", "function": {"name": "calculate"}}`** -- force a *specific* tool, bypassing selection entirely. Useful for testing one tool in isolation (exactly what Lab 2.2's "test each tool independently" step needs).

### 1.6 Parallel tool calls

When a single user turn plausibly needs multiple independent tool calls (e.g., "convert 100 USD to PKR and also tell me today's date"), a model that supports parallel tool calling can emit *multiple* `tool_call` objects in one response, each with its own `tool_call_id`. Your code executes all of them (sequentially or concurrently, your choice) and returns one `ToolMessage` per call, each tagged with the matching ID. This is a throughput optimization, not a new capability -- the same two-tool sequence could always be done as two separate model round-trips; parallel calling just collapses that into fewer round-trips when the calls don't depend on each other's results. (Calls that *do* depend on each other's results -- "convert 100 USD to PKR, then check if that's enough for the rent" -- cannot be parallelized, since the second call needs the first's output as input. That's a Day 1 prompt-chaining-shaped problem, not a parallel-tool-calling one.)

---

## 2. Building Tools: the `@tool` Decorator

### 2.1 The simplest path

`@tool` (from `langchain_core.tools`) converts a regular, type-hinted Python function into a `StructuredTool` automatically. The docstring becomes the tool's `description`; type hints generate the parameter schema. This is the LangChain abstraction layer sitting directly on top of the raw OpenAI-format schema from Section 1.3 -- you write the function once, in normal Python, and LangChain generates the wire-format JSON Schema for you.

In [5]:
@tool
def calculate(a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]) -> float:
    """Performs a single arithmetic operation between two numbers.

    Use this whenever the user asks for a numeric calculation. Do not attempt
    mental arithmetic yourself -- always call this tool for any add/subtract/
    multiply/divide request, even simple-looking ones, to guarantee precision.
    """
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b


# Inspect what @tool actually generated -- this is the schema the model sees,
# auto-derived from the type hints and docstring above.
print("Tool name:       ", calculate.name)
print("Tool description:", calculate.description)
print("Tool args schema:", json.dumps(calculate.args, indent=2))


Tool name:        calculate
Tool description: Performs a single arithmetic operation between two numbers.

    Use this whenever the user asks for a numeric calculation. Do not attempt
    mental arithmetic yourself -- always call this tool for any add/subtract/
    multiply/divide request, even simple-looking ones, to guarantee precision.
Tool args schema: {
  "a": {
    "title": "A",
    "type": "number"
  },
  "b": {
    "title": "B",
    "type": "number"
  },
  "operation": {
    "enum": [
      "add",
      "subtract",
      "multiply",
      "divide"
    ],
    "title": "Operation",
    "type": "string"
  }
}


**Notice the docstring did two jobs at once:** it's both human-readable documentation *and* the literal text the model uses to decide when to call this tool. The instruction "do not attempt mental arithmetic yourself, always call this tool" is a deliberate piece of prompt engineering embedded in the docstring -- without it, a model confident in its own arithmetic might just answer `"12 * 7 = 84"` directly in prose without calling the tool at all, which defeats the purpose of having a guaranteed-precision tool in the first place.

### 2.2 Tools with complex arguments -- `args_schema` and `StructuredTool`

The `@tool` decorator's auto-inference works well for simple flat parameters. For richer validation (the kind Day 2 built into `JobPosting`), pass an explicit Pydantic model as `args_schema` -- this gives you field descriptions, constraints, and validators on the tool's *input*, not just its output.

In [6]:
class WeatherLookupInput(BaseModel):
    city: str = Field(description="City name, e.g. 'Lahore' or 'San Francisco'")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit for the response",
    )


def _get_weather(city: str, units: Literal["celsius", "fahrenheit"] = "celsius") -> dict:
    """Mock weather lookup -- returns a fixed value so this lab doesn't need
    a real weather API key. In Thursday's session this becomes a real httpx
    call against a public API.
    """
    # deterministic mock data, keyed by lowercased city name
    mock_data = {
        "lahore": {"celsius": 38, "condition": "sunny"},
        "rawalpindi": {"celsius": 36, "condition": "partly cloudy"},
        "islamabad": {"celsius": 33, "condition": "clear"},
        "london": {"celsius": 18, "condition": "overcast"},
    }
    entry = mock_data.get(city.lower(), {"celsius": 25, "condition": "unknown (city not in mock dataset)"})
    temp_c = entry["celsius"]
    temp = temp_c if units == "celsius" else round(temp_c * 9 / 5 + 32, 1)
    return {"city": city, "temperature": temp, "units": units, "condition": entry["condition"]}


weather_tool = StructuredTool.from_function(
    func=_get_weather,
    name="get_weather",
    description="Look up the current weather for a named city. Use this for any weather-related question.",
    args_schema=WeatherLookupInput,
)

print("Tool name:       ", weather_tool.name)
print("Tool args schema:", json.dumps(weather_tool.args, indent=2))
print()
print("Direct invocation test:", weather_tool.invoke({"city": "Lahore", "units": "fahrenheit"}))


Tool name:        get_weather
Tool args schema: {
  "city": {
    "description": "City name, e.g. 'Lahore' or 'San Francisco'",
    "title": "City",
    "type": "string"
  },
  "units": {
    "default": "celsius",
    "description": "Temperature unit for the response",
    "enum": [
      "celsius",
      "fahrenheit"
    ],
    "title": "Units",
    "type": "string"
  }
}

Direct invocation test: {'city': 'Lahore', 'temperature': 100.4, 'units': 'fahrenheit', 'condition': 'sunny'}


**Why use `StructuredTool.from_function` with an explicit `args_schema` here instead of just `@tool`-decorating `_get_weather` directly?** Because `units` has a constrained set of valid values that benefits from an explicit `Field(description=...)` separate from the parameter's own docstring placement, and because separating the *input schema* (`WeatherLookupInput`) from the *implementation* (`_get_weather`) means you can unit-test the validation logic and the business logic independently -- exactly the separation-of-concerns argument from Day 1's prompt-chaining theory, applied to code structure instead of prompts.

### 2.3 `BaseTool` -- when you need full control

Subclassing `BaseTool` directly is the most verbose path, and the brief's topic list calls it out explicitly alongside `@tool` and `StructuredTool` -- worth knowing it exists and when to reach for it, even though today's lab tools don't need this level of control. Reach for `BaseTool` when you need: custom error-handling behavior beyond the default exception-to-string conversion, async-only execution with no sync fallback, or injected runtime arguments (like a database connection) that shouldn't be part of the schema the model sees.

In [7]:
class CalculatorTool(BaseTool):
    name: str = "advanced_calculate"
    description: str = "Performs arithmetic with explicit error messages for invalid operations."

    def _run(self, a: float, b: float, operation: str) -> str:
        # Full control over error handling: instead of letting an exception
        # propagate and get silently stringified by LangChain's default
        # handler, we return a controlled, model-readable error message.
        valid_ops = {"add", "subtract", "multiply", "divide"}
        if operation not in valid_ops:
            return f"ERROR: '{operation}' is not supported. Valid operations: {sorted(valid_ops)}"
        if operation == "divide" and b == 0:
            return "ERROR: division by zero is undefined"

        result = {"add": a + b, "subtract": a - b, "multiply": a * b, "divide": a / b}[operation]
        return str(result)

    async def _arun(self, a: float, b: float, operation: str) -> str:
        # No real async work happening here, but BaseTool requires this for
        # async-capable agent loops -- delegate to the sync implementation.
        return self._run(a, b, operation)


advanced_calc = CalculatorTool()
print(advanced_calc.invoke({"a": 10, "b": 0, "operation": "divide"}))
print(advanced_calc.invoke({"a": 10, "b": 3, "operation": "modulo"}))


ERROR: division by zero is undefined
ERROR: 'modulo' is not supported. Valid operations: ['add', 'divide', 'multiply', 'subtract']


Notice the deliberate design choice: invalid operations and division-by-zero return a **descriptive string**, not a raised exception. This matters specifically *because* the result flows back to the model as a `ToolMessage` -- a model that receives `"ERROR: division by zero is undefined"` as a tool result can react intelligently (apologize, ask for different inputs, explain the constraint to the user). A model that receives a raw Python traceback as a tool result generally cannot make sense of it and may hallucinate an explanation instead. This previews exactly the graceful-degradation principle Thursday's error-handling session formalizes -- today's tools are deliberately built with that habit already in place.

---

## 3. Binding Tools to a Chat Model

### 3.1 `bind_tools()`

Attaches a list of tools to a chat model, returning a new runnable that includes the tool schemas with every request. The model itself is unmodified -- `bind_tools()` returns a *new* bound object; it doesn't mutate `llm` in place.

In [8]:
tools = [calculate, weather_tool]
llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke("What's 847 multiplied by 23?")

print("Did the model call a tool?", bool(response.tool_calls))
print("Tool calls:", response.tool_calls)


Did the model call a tool? True
Tool calls: [{'name': 'calculate', 'args': {'a': 847, 'b': 23, 'operation': 'multiply'}, 'id': 'fc_193a94df-4582-434f-9f17-d335a3ab677d', 'type': 'tool_call'}]


**Reading `response.tool_calls`:** this is a list (even when there's exactly one call) of dicts, each with `name`, `args`, and `id`. Note that `response.content` at this point is typically empty or near-empty -- the model's "answer" *is* the tool call itself; it hasn't seen a result yet, so it has nothing to say in prose. This trips up beginners constantly: printing `response.content` right after a tool-call response and getting `''` is correct behavior, not a bug.

In [9]:
# Completing the full lifecycle from Section 1.2's diagram, end to end,
# by hand -- before Section 4 wraps this loop in a reusable function.

user_message = HumanMessage(content="What's 847 multiplied by 23?")
ai_response = llm_with_tools.invoke([user_message])

print("Step 1 -- model decided to call:", ai_response.tool_calls[0]["name"],
      "with args", ai_response.tool_calls[0]["args"])

# Step 2: WE execute the actual function -- the model never touched this.
tool_call = ai_response.tool_calls[0]
tool_result = calculate.invoke(tool_call["args"])
print("Step 2 -- we executed calculate() ourselves, got:", tool_result)

# Step 3: wrap the result in a ToolMessage, tagged with the matching call ID.
tool_message = ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])

# Step 4: call the model AGAIN, now with the full history including the result.
final_response = llm_with_tools.invoke([user_message, ai_response, tool_message])
print("Step 3 -- final answer after seeing the tool result:")
print(final_response.content)


Step 1 -- model decided to call: calculate with args {'a': 847, 'b': 23, 'operation': 'multiply'}
Step 2 -- we executed calculate() ourselves, got: 19481.0
Step 3 -- final answer after seeing the tool result:
847 × 23 = 19,481.


This four-step manual walkthrough is exactly what Section 1.2's diagram describes, with no abstraction hiding any step. Section 4 below wraps this into a reusable loop -- but seeing it unrolled once, by hand, is what makes the eventual loop's code legible instead of magic. If you only ever call a pre-built agent executor without having done this once, the `tool_call_id` linkage (why it exists, what breaks without it) stays opaque.

---

## 4. A Minimal Reusable Tool-Calling Loop

Section 3's four steps, generalized into a function that keeps calling tools until the model produces a final prose answer with no further tool calls. This is deliberately *not* `AgentExecutor` or a LangGraph graph -- those add retry logic, max-iteration guards, and structured state management that Thursday's error-handling session and later weeks build toward. Today's version is intentionally minimal so the control flow is fully visible.

In [10]:
def run_tool_calling_agent(user_input: str, bound_llm, tool_lookup: dict, max_turns: int = 5, verbose: bool = True) -> str:
    """Runs the tool-call loop: model decides -> we execute -> model sees
    result -> repeat until the model stops calling tools or max_turns hits.

    tool_lookup: dict mapping tool name (str) -> the actual tool object,
    so we can find the right Python callable for whatever name the model
    decided to call.
    """
    messages = [HumanMessage(content=user_input)]

    for turn in range(max_turns):
        ai_message = bound_llm.invoke(messages)
        messages.append(ai_message)

        if not ai_message.tool_calls:
            # No tool calls in this response -- model is done, has a final answer.
            return ai_message.content

        if verbose:
            print(f"--- Turn {turn + 1}: model requested {len(ai_message.tool_calls)} tool call(s) ---")

        for call in ai_message.tool_calls:
            tool_name = call["name"]
            tool_obj = tool_lookup.get(tool_name)

            if tool_obj is None:
                # Model hallucinated a tool name that doesn't exist in our
                # lookup -- this DOES happen, especially with smaller/faster
                # models under load. Don't crash; tell the model so it can
                # self-correct on the next turn instead of us raising.
                result_str = f"ERROR: tool '{tool_name}' does not exist. Available tools: {list(tool_lookup.keys())}"
            else:
                try:
                    result = tool_obj.invoke(call["args"])
                    result_str = str(result)
                except Exception as e:
                    # Same graceful-degradation principle as Section 2.3's
                    # CalculatorTool -- the model gets a readable error
                    # message, not an unhandled exception killing the loop.
                    result_str = f"ERROR executing {tool_name}: {e}"

            if verbose:
                print(f"    {tool_name}({call['args']}) -> {result_str}")

            messages.append(ToolMessage(content=result_str, tool_call_id=call["id"]))

    return "Max turns reached without a final answer -- possible tool-call loop."


In [14]:
tool_lookup = {t.name: t for t in tools}

answer = run_tool_calling_agent(
    "What's the weather in Lahore in Fahrenheit, and separately, what's 144 divided by 12?",
    llm_with_tools,
    tool_lookup,
)
print("\n=== FINAL ANSWER ===")
print(answer)


--- Turn 1: model requested 1 tool call(s) ---
    get_weather({'city': 'Lahore', 'units': 'fahrenheit'}) -> {'city': 'Lahore', 'temperature': 100.4, 'units': 'fahrenheit', 'condition': 'sunny'}
--- Turn 2: model requested 1 tool call(s) ---
    calculate({'a': 144, 'b': 12, 'operation': 'divide'}) -> 12.0

=== FINAL ANSWER ===
**Weather in Lahore (°F):** 100.4 °F, sunny  

**144 ÷ 12 =** 12.0


**Watch the turn count in the printed trace above.** If the weather lookup and the division both appear in *Turn 1* (a single turn with two tool calls listed), that's parallel tool calling in action (Section 1.6) -- the model recognized both requests were independent and batched them into one round-trip instead of two. If they appear across two separate turns instead, that's also correct behavior, just a model choosing sequential calls over parallel ones for this particular pair of requests -- not every model/prompt combination triggers parallel calling on every parallelizable request, and that's fine; the loop above handles either case identically.

---

## 5. Lab 2.2 -- Multi-Tool Research Agent (brief requirement)

**Brief text:** *"Build a 5-tool agent: `search_db`, `calculate`, `format_date`, `convert_currency`, `summarize_text`. Test each tool independently then together."*

`calculate` already exists from Section 2.1. Four more tools below, each deliberately built with a mock/deterministic backend (no external API keys needed for this lab -- Thursday's session is specifically about wiring *real* external APIs in as tools, with the error handling that requires).

In [15]:
@tool
def search_db(query: str, limit: int = 3) -> list[dict]:
    """Searches a mock internal database of company documents by keyword.

    Use this when the user asks about internal documents, reports, or
    records -- not for general knowledge questions, which don't need a
    database lookup.
    """
    # Deterministic mock corpus -- swap this for a real vector/SQL query
    # later without changing the tool's schema or calling convention.
    mock_corpus = [
        {"id": 1, "title": "Q2 2026 Security Audit Report", "snippet": "Identified 3 medium-severity findings in the payment service..."},
        {"id": 2, "title": "Onboarding Guide for New Engineers", "snippet": "All new hires must complete the security training module within..."},
        {"id": 3, "title": "Incident Postmortem: API Outage June 2026", "snippet": "Root cause was a deploy at 14:15 UTC that introduced a regression..."},
        {"id": 4, "title": "Q1 2026 Security Audit Report", "snippet": "No critical findings; 1 low-severity finding related to logging..."},
    ]
    query_lower = query.lower()
    matches = [doc for doc in mock_corpus if query_lower in doc["title"].lower() or query_lower in doc["snippet"].lower()]
    return matches[:limit]


@tool
def format_date(date_string: str, target_format: Literal["iso", "us", "long"] = "iso") -> str:
    """Reformats a date string into a target format.

    iso -> YYYY-MM-DD, us -> MM/DD/YYYY, long -> 'July 1, 2026' style.
    Use this whenever the user wants a date converted to a specific style,
    or needs a date normalized before passing it to another system.
    """
    parsed = None
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%B %d, %Y", "%d %B %Y"):
        try:
            parsed = datetime.strptime(date_string, fmt).date()
            break
        except ValueError:
            continue
    if parsed is None:
        return f"ERROR: could not parse date string '{date_string}'"

    if target_format == "iso":
        return parsed.isoformat()
    elif target_format == "us":
        return parsed.strftime("%m/%d/%Y")
    elif target_format == "long":
        return parsed.strftime("%B %-d, %Y") if os.name != "nt" else parsed.strftime("%B %d, %Y").replace(" 0", " ")


@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    """Converts an amount from one currency to another using fixed mock
    exchange rates (no live API call). Use this for any currency conversion
    question.
    """
    # Fixed mock rates relative to USD -- replace with a real FX API call
    # in a production version; the tool's calling convention stays identical.
    mock_rates_to_usd = {"USD": 1.0, "PKR": 1 / 278.5, "EUR": 1.08, "GBP": 1.26}

    from_currency = from_currency.upper()
    to_currency = to_currency.upper()

    if from_currency not in mock_rates_to_usd or to_currency not in mock_rates_to_usd:
        return {"error": f"Unsupported currency. Supported: {list(mock_rates_to_usd.keys())}"}

    usd_amount = amount * mock_rates_to_usd[from_currency]
    converted = usd_amount / mock_rates_to_usd[to_currency]
    return {
        "original_amount": amount,
        "from_currency": from_currency,
        "to_currency": to_currency,
        "converted_amount": round(converted, 2),
        "note": "Mock fixed rate, not live market data",
    }


@tool
def summarize_text(text: str, max_sentences: int = 2) -> str:
    """Produces a short extractive summary of a block of text by selecting
    its first N sentences. Use this when the user provides a long block of
    text and wants a quick summary.

    NOTE: this is a naive extractive summarizer (first-N-sentences), not an
    LLM-based abstractive summary -- it exists so this lab has a tool that
    runs with zero external dependencies. A production version would
    likely call the LLM itself for a real abstractive summary, which
    raises an interesting design question explored in the note below the
    test cases.
    """
    sentences = [s.strip() for s in text.replace("\n", " ").split(".") if s.strip()]
    summary = ". ".join(sentences[:max_sentences])
    return summary + "." if summary and not summary.endswith(".") else summary


### 5.1 Testing each tool independently (brief requirement)

Direct `.invoke()` calls, bypassing the model entirely -- this is unit testing the tool's own logic, isolated from any question about whether the model *chooses* to call it correctly. Always do this step before testing tool selection; if a tool is broken on its own, no amount of prompt tuning for tool selection will produce a correct end-to-end result.

In [16]:
print("=== search_db ===")
print(search_db.invoke({"query": "security audit", "limit": 2}))

print("\n=== calculate ===")
print(calculate.invoke({"a": 250, "b": 4, "operation": "divide"}))

print("\n=== format_date ===")
print(format_date.invoke({"date_string": "2026-07-01", "target_format": "long"}))

print("\n=== convert_currency ===")
print(convert_currency.invoke({"amount": 500, "from_currency": "USD", "to_currency": "PKR"}))

print("\n=== summarize_text ===")
sample_text = (
    "The internship started in late June 2026. Week one covered agentic AI "
    "fundamentals and a basic ReAct loop implementation. Week two introduces "
    "structured outputs and tool calling, building toward a full multi-tool "
    "agent by Friday."
)
print(summarize_text.invoke({"text": sample_text, "max_sentences": 2}))


=== search_db ===
[{'id': 1, 'title': 'Q2 2026 Security Audit Report', 'snippet': 'Identified 3 medium-severity findings in the payment service...'}, {'id': 4, 'title': 'Q1 2026 Security Audit Report', 'snippet': 'No critical findings; 1 low-severity finding related to logging...'}]

=== calculate ===
62.5

=== format_date ===
July 1, 2026

=== convert_currency ===
{'original_amount': 500.0, 'from_currency': 'USD', 'to_currency': 'PKR', 'converted_amount': 139250.0, 'note': 'Mock fixed rate, not live market data'}

=== summarize_text ===
The internship started in late June 2026. Week one covered agentic AI fundamentals and a basic ReAct loop implementation.


### 5.2 Testing all 5 tools together, bound to the model

Now the actual agent test -- the model sees all five schemas at once and has to route each part of a multi-part question to the *correct* tool, which is a meaningfully harder test than any single-tool call. This directly exercises the brief's "tool selection logic" topic.

In [17]:
all_tools = [search_db, calculate, format_date, convert_currency, summarize_text]
all_tool_lookup = {t.name: t for t in all_tools}
llm_with_all_tools = llm.bind_tools(all_tools)

multi_part_question = (
    "Find any internal documents about security audits, convert 1200 USD "
    "to PKR, and tell me what 18 times 23 is."
)

answer = run_tool_calling_agent(multi_part_question, llm_with_all_tools, all_tool_lookup)
print("\n=== FINAL ANSWER ===")
print(answer)


--- Turn 1: model requested 1 tool call(s) ---
    search_db({'limit': 5, 'query': 'security audits'}) -> []
--- Turn 2: model requested 1 tool call(s) ---
    search_db({'limit': 5, 'query': 'security audit'}) -> [{'id': 1, 'title': 'Q2 2026 Security Audit Report', 'snippet': 'Identified 3 medium-severity findings in the payment service...'}, {'id': 4, 'title': 'Q1 2026 Security Audit Report', 'snippet': 'No critical findings; 1 low-severity finding related to logging...'}]
--- Turn 3: model requested 1 tool call(s) ---
    convert_currency({'amount': 1200, 'from_currency': 'USD', 'to_currency': 'PKR'}) -> {'original_amount': 1200.0, 'from_currency': 'USD', 'to_currency': 'PKR', 'converted_amount': 334200.0, 'note': 'Mock fixed rate, not live market data'}
--- Turn 4: model requested 1 tool call(s) ---
    calculate({'a': 18, 'b': 23, 'operation': 'multiply'}) -> 414.0

=== FINAL ANSWER ===
**Internal security‑audit documents found**

| ID | Title | Brief snippet |
|----|-------|-----

**Things to actually verify against the trace above, not just that it ran:**
- Did the model call `search_db` with a query term that would actually match the mock corpus (e.g., `"security audit"`), or did it pass through the user's exact phrasing including words like "internal documents" that aren't in any document title/snippet? This is the tool-selection-vs-argument-quality distinction -- calling the *right* tool with *wrong* arguments is a different failure mode than calling the wrong tool, and the trace should let you tell which happened if results come back empty.
- Did `convert_currency` get called with `from_currency="USD"` and `to_currency="PKR"` in the correct order, or swapped? Argument-order confusion is a real, observed failure mode when a tool has two same-typed parameters (`from_currency`/`to_currency` are both strings) -- this is exactly the kind of thing a stricter `args_schema` (with per-field descriptions reinforcing direction) reduces but doesn't eliminate.
- Did all three sub-tasks get routed to tools at all, or did the model answer the arithmetic part directly in prose without calling `calculate`? Compare against Section 2.1's docstring instruction ("always call this tool, even for simple-looking requests") -- if arithmetic skipped the tool, that instruction wasn't strong enough for this particular phrasing, which is itself a useful, concrete finding to bring to Friday's standup blockers discussion.

### 5.3 A genuinely useful design question, surfaced by `summarize_text`

`summarize_text`'s docstring flagged something worth sitting with rather than skipping past: it's a non-LLM, naive extractive summarizer, deliberately built that way so this lab needs zero extra API calls. But a *real* summarization tool, in an agent that already has an LLM available, raises a real architectural question -- **should "summarize" even be a tool at all, or should it just be something the agent's own reasoning does directly in a final response, with no tool call required?** There's no single correct answer; it depends on whether you need summarization to be independently *callable, testable, and swappable* (tool) versus whether it's just part of the agent's natural language generation (no tool needed). This exact question -- "tool vs. just let the LLM do it directly" -- is explicitly Friday's "Architecture Review" agenda item ("when to use tools vs chains?"), so it's worth having a concrete opinion on by then, grounded in something you actually built this week rather than an abstract guess.

---

## 6. Self-Check Before Thursday

1. Walk through, in your own words, why a model emitting a `tool_call` is fundamentally different from a model executing code -- specifically, what security property does this separation give you that direct code execution wouldn't?
2. In Section 4's loop, what happens if the model calls a tool name that isn't in `tool_lookup`? Why does the loop choose to feed an error string back to the model instead of raising an exception and stopping?
3. You bound 5 tools to the model in Section 5.2. If you'd only bound 2 of those 5 (say, just `calculate` and `convert_currency`) and asked the same multi-part question, what would you expect to happen to the `search_db`-shaped part of the question? Run this experiment for real, not just in your head -- it's a fast way to build intuition for what "tool selection" can and can't do when the right tool simply isn't available.

Tomorrow (Thursday) replaces every mock backend built today with a real external API call -- `search_db`'s in-memory list becomes an actual database query or live search API, and the error-handling section's graceful-degradation patterns (already previewed today in `CalculatorTool._run` and the agent loop's try/except) get formalized into retry logic with exponential backoff for things that can genuinely fail over a network: timeouts, rate limits, malformed API responses. Today's tools never fail in interesting ways because they're deterministic mocks; tomorrow's will, and that's the point.
